# Setup

In [1]:
import os, sys
from pathlib import Path
PROJECT = Path.home() / "Desktop" / "Project" / "sales-forecasting-assistant"
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
Path("src/rag").mkdir(parents=True, exist_ok=True)
print(os.getcwd())

/Users/dayoawe1/Desktop/Project/sales-forecasting-assistant


### Install the OpenAI library and add your key safely

In [ ]:
%pip install openai

### Forecast summaries (the documents the RAG step retrieves)

In [2]:
%%writefile src/rag/summaries.py
"""Turn the latest forecasts into plain-English summaries (documents for RAG)."""
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas
from src.ingest.load_raw import get_connection

QUERY = """
WITH latest AS (
  SELECT * FROM SALES_DB.ANALYTICS.FORECASTS
  WHERE RUN_TS = (SELECT MAX(RUN_TS) FROM SALES_DB.ANALYTICS.FORECASTS)
)
SELECT f.STORE_NBR, f.FAMILY, f.WEEK_START, f.FORECAST,
       x.SALES_LAG_1 AS LAST_WEEK, x.SALES_LAG_52 AS SAME_WEEK_LAST_YEAR,
       x.SALES_ROLL_MEAN_4 AS AVG_4WK, x.PROMO_ITEMS, x.HOLIDAY_DAYS,
       s.CITY, s.STATE, s.STORE_TYPE
FROM latest f
JOIN SALES_DB.ANALYTICS.FEATURES x
  ON x.STORE_NBR = f.STORE_NBR AND x.FAMILY = f.FAMILY AND x.WEEK_START = f.WEEK_START
JOIN SALES_DB.STAGING.STORES s ON s.STORE_NBR = f.STORE_NBR
"""


def _pct(new, old):
    return None if not old or pd.isna(old) else (new - old) / old * 100


def _fmt(p):
    if p is None or pd.isna(p):
        return "n/a"
    return f"{'up' if p >= 0 else 'down'} {abs(p):.1f}%"


def _family_moves(g, n=3):
    d = g.assign(CHANGE=g["FORECAST"] - g["AVG_4WK"]).dropna(subset=["CHANGE"])
    line = lambda r: f"{r.FAMILY} ({r.FORECAST:,.0f} vs {r.AVG_4WK:,.0f}, {_fmt(_pct(r.FORECAST, r.AVG_4WK))})"
    up = "; ".join(line(r) for r in d.nlargest(n, "CHANGE").itertuples())
    down = "; ".join(line(r) for r in d.nsmallest(n, "CHANGE").itertuples())
    return up, down


def build_summaries():
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute(QUERY)
        df = cur.fetch_pandas_all()
    finally:
        conn.close()

    num = ["FORECAST", "LAST_WEEK", "SAME_WEEK_LAST_YEAR", "AVG_4WK", "PROMO_ITEMS", "HOLIDAY_DAYS"]
    df[num] = df[num].apply(pd.to_numeric, errors="coerce")
    df["HOLIDAY_DAYS"] = df["HOLIDAY_DAYS"].fillna(0)
    week = str(pd.to_datetime(df["WEEK_START"]).max().date())

    docs = []
    for store, g in df.groupby("STORE_NBR"):
        fc, lw = g["FORECAST"].sum(), g["LAST_WEEK"].sum()
        ly, a4 = g["SAME_WEEK_LAST_YEAR"].sum(), g["AVG_4WK"].sum()
        up, down = _family_moves(g)
        first = g.iloc[0]
        text = (f"Store {int(store)} ({first['CITY']}, {first['STATE']}; store type {first['STORE_TYPE']}) - "
                f"forecast for the week of {week}. "
                f"Total forecast sales: {fc:,.0f}, {_fmt(_pct(fc, lw))} vs last week ({lw:,.0f}), "
                f"{_fmt(_pct(fc, a4))} vs its 4-week average ({a4:,.0f}), and "
                f"{_fmt(_pct(fc, ly))} vs the same week last year ({ly:,.0f}). "
                f"Largest expected increases vs the 4-week average: {up}. "
                f"Largest expected decreases: {down}. "
                f"Product families with items on promotion last week: {int((g['PROMO_ITEMS'] > 0).sum())} of {len(g)}. "
                f"Holiday days at this store during the forecast week: {int(g['HOLIDAY_DAYS'].max())}.")
        docs.append({"STORE_NBR": int(store), "WEEK_START": week, "SUMMARY": text})

    # one chain-wide document
    t = df.groupby("STORE_NBR").agg(FC=("FORECAST", "sum"), LW=("LAST_WEEK", "sum"))
    t["PCT"] = (t["FC"] - t["LW"]) / t["LW"] * 100
    ups = ", ".join(f"store {s} ({_fmt(p)})" for s, p in t["PCT"].nlargest(5).items())
    downs = ", ".join(f"store {s} ({_fmt(p)})" for s, p in t["PCT"].nsmallest(5).items())
    total, total_lw = df["FORECAST"].sum(), df["LAST_WEEK"].sum()
    docs.append({"STORE_NBR": 0, "WEEK_START": week, "SUMMARY":
                 f"Chain-wide forecast for the week of {week}: total forecast sales {total:,.0f} across "
                 f"{df['STORE_NBR'].nunique()} stores and {df['FAMILY'].nunique()} product families, "
                 f"{_fmt(_pct(total, total_lw))} vs last week ({total_lw:,.0f}). "
                 f"Stores with the largest expected increase vs last week: {ups}. "
                 f"Stores with the largest expected decrease vs last week: {downs}."})

    out = pd.DataFrame(docs)
    conn = get_connection()
    try:
        ok, _, n, _ = write_pandas(conn, out, table_name="STORE_SUMMARIES", database="SALES_DB",
                                   schema="ANALYTICS", auto_create_table=True,
                                   overwrite=True, quote_identifiers=False)
    finally:
        conn.close()
    print(f"STORE_SUMMARIES {'OK' if ok else 'FAILED'} {n} rows")
    return out

Overwriting src/rag/summaries.py


### The vector index (embeddings + similarity search)

In [3]:
%%writefile src/rag/index.py
"""Embed the store summaries with OpenAI and search them by meaning."""
import json
import os
from pathlib import Path
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from src.ingest.load_raw import get_connection

load_dotenv()
INDEX_DIR = Path(__file__).resolve().parents[2] / "data" / "rag"
EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-3-small")


def embed(texts):
    resp = OpenAI().embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)   # normalize for cosine similarity


def build_index():
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT STORE_NBR, WEEK_START, SUMMARY FROM SALES_DB.ANALYTICS.STORE_SUMMARIES ORDER BY STORE_NBR")
        rows = cur.fetchall()
    finally:
        conn.close()

    docs = [{"store_nbr": int(r[0]), "week_start": str(r[1]), "text": r[2]} for r in rows]
    vecs = embed([d["text"] for d in docs])
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    np.save(INDEX_DIR / "vectors.npy", vecs)
    (INDEX_DIR / "docs.json").write_text(json.dumps(docs))
    print(f"Indexed {len(docs)} documents with {EMBED_MODEL}")


def search(question, k=3):
    vecs = np.load(INDEX_DIR / "vectors.npy")
    docs = json.loads((INDEX_DIR / "docs.json").read_text())
    scores = vecs @ embed([question])[0]
    top = np.argsort(scores)[::-1][:k]
    return [{**docs[i], "score": round(float(scores[i]), 3)} for i in top]


def refresh():
    """Pipeline entry point: rebuild summaries, then re-embed them."""
    from src.rag.summaries import build_summaries
    build_summaries()
    build_index()

Overwriting src/rag/index.py


### The assistant (an agent with two tools)

In [4]:
%%writefile src/rag/assistant.py
"""LLM assistant: answers forecast questions with read-only SQL or retrieved summaries."""
import json
import os
import re
import snowflake.connector
from openai import OpenAI
from dotenv import load_dotenv
from src.rag.index import search

load_dotenv()
CHAT_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")

ALLOWED_TABLES = {
    "SALES_DB.ANALYTICS.FORECASTS", "SALES_DB.ANALYTICS.FEATURES",
    "SALES_DB.ANALYTICS.ACCURACY_LOG", "SALES_DB.ANALYTICS.MODEL_RESULTS",
    "SALES_DB.ANALYTICS.STORE_SUMMARIES", "SALES_DB.STAGING.STORES",
    "SALES_DB.STAGING.SALES_WEEKLY",
}

SCHEMA = """
SALES_DB.ANALYTICS.FORECASTS(STORE_NBR, FAMILY, WEEK_START, FORECAST, MODEL, RUN_TS)
  -- next-week forecasts, one row per store x family per pipeline run. ALWAYS filter to the latest run:
  -- WHERE RUN_TS = (SELECT MAX(RUN_TS) FROM SALES_DB.ANALYTICS.FORECASTS)
SALES_DB.ANALYTICS.FEATURES(STORE_NBR, FAMILY, WEEK_START, STORE_TYPE, STORE_CLUSTER, SALES, PROMO_ITEMS,
  HOLIDAY_DAYS, AVG_OIL_PRICE, STORE_TRANSACTIONS, IS_FUTURE, SALES_LAG_1, SALES_LAG_2, SALES_LAG_4,
  SALES_LAG_52, SALES_ROLL_MEAN_4, SALES_ROLL_MEAN_12, SALES_ROLL_STD_12, WEEK_OF_YEAR, MONTH, YEAR)
  -- weekly history. IS_FUTURE = 1 is the forecast week (SALES is NaN there); use IS_FUTURE = 0 for actual sales.
  -- For the forecast week, SALES_LAG_1 = last week's actual sales, SALES_LAG_52 = same week last year.
  -- Lag/rolling columns can be NaN in early weeks.
SALES_DB.ANALYTICS.ACCURACY_LOG(RUN_TS, MODEL, EVAL_WEEK, WAPE, N_SERIES, THRESHOLD, PASSED)
  -- one row per pipeline quality check; WAPE is a fraction (0.0999 = 9.99%).
SALES_DB.ANALYTICS.MODEL_RESULTS(MODEL, SCOPE, N, WAPE, RMSE)
  -- backtest comparison of naive_52, arima_fourier, prophet, xgboost_global; SCOPE is 'all' or 'top10'.
SALES_DB.ANALYTICS.STORE_SUMMARIES(STORE_NBR, WEEK_START, SUMMARY)
SALES_DB.STAGING.STORES(STORE_NBR, CITY, STATE, STORE_TYPE, STORE_CLUSTER)
SALES_DB.STAGING.SALES_WEEKLY(STORE_NBR, FAMILY, WEEK_START, CITY, STATE, STORE_TYPE, STORE_CLUSTER,
  SALES, PROMO_ITEMS, HOLIDAY_DAYS, AVG_OIL_PRICE, STORE_TRANSACTIONS, DAYS_IN_WEEK)
"""

SYSTEM = f"""You are a sales forecasting assistant for a grocery chain in Ecuador
(54 stores, 33 product families). Answer ONLY from tool results; never invent numbers.
- Use run_sql for numbers, rankings, totals and comparisons. Use fully qualified table names.
- Use search_summaries for "why" or narrative questions about a store's or the chain's forecast.
- Be concise: lead with the answer, include store numbers and figures, and say which data you used.
Tables:
{SCHEMA}"""

TOOLS = [
    {"type": "function", "function": {
        "name": "run_sql",
        "description": "Run ONE read-only Snowflake SELECT query and return up to 100 rows.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "search_summaries",
        "description": "Retrieve the most relevant written forecast summaries (per store or chain-wide).",
        "parameters": {"type": "object", "properties": {"question": {"type": "string"}},
                       "required": ["question"]}}},
]

BLOCKED = r"\b(insert|update|delete|merge|drop|create|alter|truncate|grant|revoke|copy|put|call|use)\b"


def _ro_connection():
    return snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        private_key_file=os.path.expanduser(os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]),
        role="ANALYST_RO", warehouse="SALES_WH", database="SALES_DB")


def run_sql(query):
    q = query.strip().rstrip(";")
    if ";" in q:
        return {"error": "Only one statement is allowed."}
    if not re.match(r"(?is)^\s*(select|with)\b", q) or re.search(BLOCKED, q, re.I):
        return {"error": "Only read-only SELECT queries are allowed."}
    ctes = {c.upper() for c in re.findall(r"(?i)\b(\w+)\s+as\s*\(", q)}
    for t in re.findall(r"(?i)\b(?:from|join)\s+([A-Za-z0-9_.\"]+)", q):
        name = t.replace('"', "").upper()
        if name not in ALLOWED_TABLES and name not in ctes:
            return {"error": f"Table {t} is not allowed. Use fully qualified names from the schema."}
    if not re.search(r"(?i)\blimit\s+\d+\s*$", q):
        q += " LIMIT 100"
    conn = _ro_connection()
    try:
        cur = conn.cursor()
        cur.execute(q)
        df = cur.fetch_pandas_all()
        return {"rows": json.loads(df.head(100).to_json(orient="records", date_format="iso"))}
    except Exception as e:
        return {"error": str(e)[:500]}
    finally:
        conn.close()


def ask(question, verbose=True, max_steps=6):
    client = OpenAI()
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = client.chat.completions.create(model=CHAT_MODEL, messages=messages,
                                             tools=TOOLS, temperature=0).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = run_sql(**args) if tc.function.name == "run_sql" else search(args["question"])
            if verbose:
                print(f"  [{tc.function.name}] {list(args.values())[0][:200]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, default=str)[:12000]})
    return "Sorry, I couldn't complete that within the step limit."

Overwriting src/rag/assistant.py


### Build the summaries and the index

In [5]:
from src.rag.index import refresh
refresh()

from src.rag.index import search
for hit in search("Why is store 44's forecast lower than usual?"):
    print(hit["score"], "|", hit["text"][:300], "\n")

STORE_SUMMARIES OK 55 rows
Indexed 55 documents with text-embedding-3-small
0.585 | Store 38 (Loja, Loja; store type D) - forecast for the week of 2017-08-14. Total forecast sales: 79,759, up 3.8% vs last week (76,817), down 13.1% vs its 4-week average (91,800), and up 2.5% vs the same week last year (77,789). Largest expected increases vs the 4-week average: DAIRY (5,552 vs 5,436, 

0.56 | Chain-wide forecast for the week of 2017-08-14: total forecast sales 5,599,503 across 54 stores and 33 product families, up 4.0% vs last week (5,385,402). Stores with the largest expected increase vs last week: store 44 (up 11.6%), store 24 (up 10.9%), store 4 (up 10.8%), store 8 (up 10.7%), store 1 

0.553 | Store 33 (Quevedo, Los Rios; store type C) - forecast for the week of 2017-08-14. Total forecast sales: 67,838, down 0.6% vs last week (68,259), down 2.0% vs its 4-week average (69,234), and up 5.7% vs the same week last year (64,184). Largest expected increases vs the 4-week average: LIQUOR,WI

### Ask questions

In [6]:
from src.rag.assistant import ask

for q in ["Which 5 stores have the highest total forecast for next week?",
          "Which product family has the largest forecast next week, and how does it compare to last week?",
          "Why is store 44's forecast different from its recent average?",
          "Did the latest model run pass the quality gate?"]:
    print("Q:", q)
    print("A:", ask(q), "\n" + "-" * 80)

Q: Which 5 stores have the highest total forecast for next week?
  [run_sql] SELECT STORE_NBR, SUM(FORECAST) AS TOTAL_FORECAST
FROM SALES_DB.ANALYTICS.FORECASTS
WHERE RUN_TS = (SELECT MAX(RUN_TS) FROM SALES_DB.ANALYTICS.FORECASTS)
GROUP BY STORE_NBR
ORDER BY TOTAL_FORECAST DES
A: The 5 stores with the highest total forecast for next week are:
1. Store 44 with 298,828 units
2. Store 45 with 271,561 units
3. Store 47 with 253,993 units
4. Store 3 with 234,701 units
5. Store 49 with 230,653 units

This is based on the latest forecast run data. 
--------------------------------------------------------------------------------
Q: Which product family has the largest forecast next week, and how does it compare to last week?
  [run_sql] WITH latest_forecast AS (
  SELECT FAMILY, SUM(FORECAST) AS total_forecast
  FROM SALES_DB.ANALYTICS.FORECASTS
  WHERE RUN_TS = (SELECT MAX(RUN_TS) FROM SALES_DB.ANALYTICS.FORECASTS)
  GROUP BY FAMILY
A: The product family with the largest forecast next week is

### Measure accuracy (the evaluation most projects skip)

In [7]:
%%writefile src/rag/evaluate_assistant.py
"""Score the assistant against SQL-computed ground truth."""
import re
from src.ingest.load_raw import get_connection
from src.rag.assistant import ask

LATEST = "(SELECT MAX(RUN_TS) FROM SALES_DB.ANALYTICS.FORECASTS)"
CASES = [
    ("Which 5 stores have the highest total forecast for next week?",
     f"SELECT STORE_NBR FROM SALES_DB.ANALYTICS.FORECASTS WHERE RUN_TS = {LATEST} "
     "GROUP BY STORE_NBR ORDER BY SUM(FORECAST) DESC LIMIT 5", "ints"),
    ("Which product family has the largest total forecast next week?",
     f"SELECT FAMILY FROM SALES_DB.ANALYTICS.FORECASTS WHERE RUN_TS = {LATEST} "
     "GROUP BY FAMILY ORDER BY SUM(FORECAST) DESC LIMIT 1", "text"),
    ("What was the WAPE in the most recent model accuracy check?",
     "SELECT WAPE FROM SALES_DB.ANALYTICS.ACCURACY_LOG ORDER BY RUN_TS DESC LIMIT 1", "number"),
    ("Which model had the lowest WAPE on the top 10 series in the backtest?",
     "SELECT MODEL FROM SALES_DB.ANALYTICS.MODEL_RESULTS WHERE SCOPE = 'top10' ORDER BY WAPE LIMIT 1", "text"),
    ("How many store and product family combinations have a forecast for next week?",
     f"SELECT COUNT(*) FROM SALES_DB.ANALYTICS.FORECASTS WHERE RUN_TS = {LATEST}", "number"),
    ("Which city has the most stores?",
     "SELECT CITY FROM SALES_DB.STAGING.STORES GROUP BY CITY ORDER BY COUNT(*) DESC LIMIT 1", "text"),
    ("What is the total forecast across all stores for next week?",
     f"SELECT SUM(FORECAST) FROM SALES_DB.ANALYTICS.FORECASTS WHERE RUN_TS = {LATEST}", "number"),
    ("Which 3 stores have the biggest forecasted percentage decline versus last week?",
     "SELECT f.STORE_NBR FROM SALES_DB.ANALYTICS.FORECASTS f JOIN SALES_DB.ANALYTICS.FEATURES x "
     "ON x.STORE_NBR = f.STORE_NBR AND x.FAMILY = f.FAMILY AND x.WEEK_START = f.WEEK_START "
     f"WHERE f.RUN_TS = {LATEST} GROUP BY f.STORE_NBR "
     "ORDER BY (SUM(f.FORECAST) - SUM(x.SALES_LAG_1)) / NULLIF(SUM(x.SALES_LAG_1), 0) ASC LIMIT 3", "ints"),
    ("Did the latest model run pass the quality gate?",
     "SELECT PASSED FROM SALES_DB.ANALYTICS.ACCURACY_LOG ORDER BY RUN_TS DESC LIMIT 1", "yesno"),
]


def _numbers(text):
    return [float(n) for n in re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))]


def _check(kind, truth, answer):
    a = answer.lower()
    if kind == "ints":
        return {int(v) for v in truth} <= {int(n) for n in _numbers(answer) if n.is_integer()}
    if kind == "text":
        return str(truth[0]).split("_")[0].lower() in a
    if kind == "number":
        v = float(truth[0])
        return any(abs(n - c) <= 0.015 * abs(c) for n in _numbers(answer)
                   for c in (v, v * 100, v / 1e3, v / 1e6) if c)
    if kind == "yesno":
        return ("pass" in a or "yes" in a) if truth[0] else ("fail" in a or "no" in a)


def main():
    conn = get_connection()
    cur = conn.cursor()
    passed = 0
    for question, truth_sql, kind in CASES:
        cur.execute(truth_sql)
        truth = [r[0] for r in cur.fetchall()]
        answer = ask(question, verbose=False)
        ok = _check(kind, truth, answer)
        passed += ok
        print(f"{'PASS' if ok else 'FAIL'} | {question}\n       truth: {truth}\n       answer: {answer[:250]}\n")
    conn.close()
    print(f"Assistant accuracy: {passed}/{len(CASES)} = {passed / len(CASES):.0%}")
    return passed / len(CASES)

Writing src/rag/evaluate_assistant.py


In [8]:
from src.rag.evaluate_assistant import main as eval_assistant
eval_assistant()

PASS | Which 5 stores have the highest total forecast for next week?
       truth: [44, 45, 47, 3, 49]
       answer: The 5 stores with the highest total forecast for next week are:
1. Store 44 with a forecast of 298,828
2. Store 45 with a forecast of 271,561
3. Store 47 with a forecast of 253,993
4. Store 3 with a forecast of 234,701
5. Store 49 with a forecast of 

PASS | Which product family has the largest total forecast next week?
       truth: ['GROCERY I']
       answer: The product family with the largest total forecast next week is "GROCERY I" with a total forecast of 1,647,241.85. This is based on the latest forecast run data.

PASS | What was the WAPE in the most recent model accuracy check?
       truth: [0.10366496616425283]
       answer: The WAPE in the most recent model accuracy check was approximately 10.37%, based on the latest data from the accuracy log.

PASS | Which model had the lowest WAPE on the top 10 series in the backtest?
       truth: ['xgboost_global']
   

0.8888888888888888

### Add the RAG refresh to the Airflow pipeline

#### update the DAG, requirements, and compose file

In [9]:
dag = Path("dags/sales_forecasting_dag.py")
s = dag.read_text()
if "_refresh_rag" not in s:
    s = s.replace("\n\ndefault_args", '''

def _refresh_rag():
    from src.rag.index import refresh
    refresh()


default_args''')
    s = s.replace("    load_raw >> build_staging", '''    refresh_rag = PythonOperator(task_id="refresh_rag", python_callable=_refresh_rag)

    load_raw >> build_staging''')
    s = s.replace(">> forecast_next_week\n", ">> forecast_next_week >> refresh_rag\n")
    dag.write_text(s)

req = Path("requirements-airflow.txt")
if "openai" not in req.read_text():
    req.write_text(req.read_text().rstrip("\n") + "\nopenai\n")

comp = Path("docker-compose.yml")
c = comp.read_text()
if "OPENAI_API_KEY" not in c:
    c = c.replace('      WAPE_THRESHOLD: "0.12"\n',
                  '      WAPE_THRESHOLD: "0.12"\n      OPENAI_API_KEY: ${OPENAI_API_KEY}\n')
    comp.write_text(c)

print(dag.read_text()[-400:])
print(req.read_text())

or(task_id="evaluate_model", python_callable=_evaluate_model)
    forecast_next_week = PythonOperator(task_id="forecast_next_week",
                                        python_callable=_forecast_next_week)

    refresh_rag = PythonOperator(task_id="refresh_rag", python_callable=_refresh_rag)

    load_raw >> build_staging >> build_features >> evaluate_model >> forecast_next_week >> refresh_rag

pyspark==3.5.3
xgboost==2.1.1
python-dotenv
scikit-learn
openai



### hybrid retrieval

In [10]:
%%writefile src/rag/index.py
"""Embed the store summaries with OpenAI and search them (hybrid: store filter + semantic)."""
import json
import os
import re
from pathlib import Path
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from src.ingest.load_raw import get_connection

load_dotenv()
INDEX_DIR = Path(__file__).resolve().parents[2] / "data" / "rag"
EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-3-small")


def embed(texts):
    resp = OpenAI().embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)


def build_index():
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT STORE_NBR, WEEK_START, SUMMARY FROM SALES_DB.ANALYTICS.STORE_SUMMARIES ORDER BY STORE_NBR")
        rows = cur.fetchall()
    finally:
        conn.close()
    docs = [{"store_nbr": int(r[0]), "week_start": str(r[1]), "text": r[2]} for r in rows]
    vecs = embed([d["text"] for d in docs])
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    np.save(INDEX_DIR / "vectors.npy", vecs)
    (INDEX_DIR / "docs.json").write_text(json.dumps(docs))
    print(f"Indexed {len(docs)} documents with {EMBED_MODEL}")


def search(question, k=3):
    """Hybrid retrieval: stores named in the question first, then best semantic matches."""
    vecs = np.load(INDEX_DIR / "vectors.npy")
    docs = json.loads((INDEX_DIR / "docs.json").read_text())
    named = {int(n) for n in re.findall(r"(?i)\bstores?\s*(?:#|no\.?|number)?\s*(\d+)", question)}
    scores = vecs @ embed([question])[0]
    order = sorted(range(len(docs)),
                   key=lambda i: (docs[i]["store_nbr"] not in named, -scores[i]))
    return [{**docs[i], "score": round(float(scores[i]), 3),
             "matched_by": "store filter" if docs[i]["store_nbr"] in named else "semantic"}
            for i in order[:k]]


def refresh():
    """Pipeline entry point: rebuild summaries, then re-embed them."""
    from src.rag.summaries import build_summaries
    build_summaries()
    build_index()

Overwriting src/rag/index.py


### define percentage change in the assistant's instructions.

In [11]:
from pathlib import Path
p = Path("src/rag/assistant.py")
s = p.read_text()
rule = ("- Percentage change for a store, family or the chain = (SUM(new) - SUM(old)) / SUM(old) "
        "computed on totals, never an average of row-level percentages. For 'vs last week' on the "
        "forecast week, old = SUM(SALES_LAG_1) from FEATURES joined on STORE_NBR, FAMILY, WEEK_START.\n")
if "computed on totals" not in s:
    s = s.replace("- Be concise:", rule + "- Be concise:")
    p.write_text(s)
print("Rule added" if "computed on totals" in p.read_text() else "Not added - check file")

Rule added


Do a restart and run cell 1 again before running the next cell

In [2]:
from src.rag.index import search
for hit in search("Why is store 44's forecast different from its recent average?"):
    print(hit["matched_by"], hit["score"], "|", hit["text"][:200], "\n")

from src.rag.evaluate_assistant import main as eval_assistant
eval_assistant()

store filter 0.557 | Store 44 (Quito, Pichincha; store type A) - forecast for the week of 2017-08-14. Total forecast sales: 298,828, up 11.6% vs last week (267,854), down 1.7% vs its 4-week average (303,857), and up 2.1%  

semantic 0.601 | Store 38 (Loja, Loja; store type D) - forecast for the week of 2017-08-14. Total forecast sales: 79,759, up 3.8% vs last week (76,817), down 13.1% vs its 4-week average (91,800), and up 2.5% vs the sa 

semantic 0.586 | Chain-wide forecast for the week of 2017-08-14: total forecast sales 5,599,503 across 54 stores and 33 product families, up 4.0% vs last week (5,385,402). Stores with the largest expected increase vs  

PASS | Which 5 stores have the highest total forecast for next week?
       truth: [44, 45, 47, 3, 49]
       answer: The 5 stores with the highest total forecast for next week are store 44 with 298,828, store 45 with 271,561, store 47 with 253,993, store 3 with 234,701, and store 49 with 230,653. This is based on the latest forecast 

1.0